# RL training for LLM agents — 3. Training real agents

### Multi-turn tool use, recording an unmodified agent, and verl + Agent Lightning end to end

| # | Notebook | What it builds | Sections |
|---|---|---|---|
| 1 | [GRPO from scratch](1_grpo_from_scratch.ipynb) | an RL trainer in plain PyTorch, switched to verl's algorithm functions once they're checked against ours, then a real inference engine (vLLM) and weight sync | §0–§2 |
| 2 | [From our trainer to verl](2_from_trainer_to_verl.ipynb) | the same trainer, rebuilt from verl's pieces: `DataProto`, Ray actors, worker groups; then plain verl | §3, §5–§6 |
| 3 | [Training real agents](3_training_real_agents.ipynb) | multi-turn tool use, recording a real agent at the HTTP boundary, Agent Lightning, and verl + Agent Lightning end to end | §7–§10 |
| 4 | [Rewards: exact checks and LLM judges](4_rewards_and_judges.ipynb) | what a reward measures, and an LLM judge next to an exact check | own numbering |
| 5 | [Calc-X end to end](5_calc_x_end_to_end.ipynb) | one training step driven by hand through Agent Lightning, then a real training run | own numbering |

Notebooks 1–3 share one set of section numbers, so a reference like "§1.5" or "§8.1" always points to the same place, whichever notebook it appears in. Each notebook runs on its own: code from earlier notebooks is imported from `tutorial/lib/`, where it is copied verbatim.

**Where we are.** [Notebook 1](1_grpo_from_scratch.ipynb) built a GRPO trainer from scratch, switched it to verl's algorithm functions, and added vLLM for generation. [Notebook 2](2_from_trainer_to_verl.ipynb) rebuilt it from verl's pieces (`DataProto`, Ray actors, worker groups) and ran plain verl on our arithmetic task. All of that was **single-turn**: one prompt in, one response out.

**In this notebook** the model becomes an agent:

| § | We write | Then replace it with | The problem that forces the swap |
|---|---|---|---|
| 7 | a token-level multi-turn tool loop | verl's `ToolAgentLoop`, then… a **real** agent | real agents speak HTTP and own their control flow |
| 8 | a 60-line HTTP gateway that records tokens | Agent Lightning's Gateway + Controller | rollouts as processes, state, failures, merging calls into rows |
| 9 | | plain `verl` and `verl` + **Agent Lightning**, end to end | |

**Where Agent Lightning fits.** For agents we're willing to (re)write as a verl agent loop, verl alone is the right tool (§7.2, §9.1). [Agent Lightning](https://github.com/microsoft/agent-lightning) starts paying off the moment the agent is an existing program we don't want to rewrite (§7.3). This isn't a tutorial on its API (its [docs](https://microsoft.github.io/agent-lightning/) cover that); we build the idea ourselves in §8, then swap in the real thing and look at what it does that our version doesn't.

**What we need to know already.** Notebook 1's trainer (§1.3–§2.4) and notebook 2's verl helpers (§3); both are imported here from `tutorial/lib/`.

**Hardware and time.** One GPU with about 80 GB, and about 10 minutes on an H100 (plus a 3 GB download for the 1.5B model used from §8 on).

## 0. Setup

The same setup as the other notebooks, plus notebook 1's trainer (`rl_core`) and notebook 2's verl helpers (`verl_core`), imported from `tutorial/lib/`.

In [ ]:
import gc
import json
import os
import random
import re
import socket
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "calc_x").is_dir())
sys.path.insert(0, str(REPO / "tutorial/lib"))  # the series' shared code
from common import (CALC_X_DATA, CALC_X_MODEL, DEVICE, GENERATED, MODEL, SEED, SERVICES, SHM, VENV_BIN, WORK,  # noqa: E402
                    follow_log, gpu_free_gib, log_tail, setup, start_service, stop_service, wait_until_healthy)

setup()

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from rl_core import (EVAL, INSTRUCTION, LR, PAD_ID, STOP_IDS, TRAIN, chat_prompt_ids, compute_log_probs,  # noqa: E402
                     decode_tokens, evaluate, grpo_advantages, load_policy, make_optimizer, make_padded_batch,
                     plot_histories, ppo_clip_loss, ppo_update, print_step_metrics, push_weights, rl_step,
                     save_for_inference, select_rows, tokenizer, vllm_generate)
from arith_task import BOXED, make_problems, reward_fn, safe_eval  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402
from vllm import LLM, SamplingParams, TokensPrompt  # noqa: E402
from verl import DataProto  # noqa: E402
from verl_core import add_advantages, build_batch, compute_batch_metrics, verl_ppo_loss  # noqa: E402
import httpx  # noqa: E402

§7 trains on multi-turn tool episodes with the setup from the end of notebook 1: the policy and its optimizer in this process, and a vLLM engine sharing the GPU. (Notebook 2 moved these into Ray actors; the tool loop works the same either way, so we keep it simple here.)

In [ ]:
policy = load_policy()
optimizer = make_optimizer(policy, LR)
llm = LLM(model=MODEL, gpu_memory_utilization=0.3, max_model_len=4096, seed=SEED)
step_rng = random.Random(SEED)
print(f"GPU free: {gpu_free_gib():.0f} GiB")

## 7. Multi-turn: agents that call tools

So far, one prompt in, one response out. Agents don't work like that: the model writes a tool call, something executes it, the result goes back into the context, and the model continues. The training data for one attempt becomes a single sequence where some tokens came from the model and some from the **environment** (the tools):

```
[ prompt ][ model: <tool_call>... ][ env: <tool_response>... ][ model: ... \boxed{N} ]
            response_mask = 1          response_mask = 0            response_mask = 1
```

**Only the model's tokens are trained.** The tool's output stays in the sequence, because the model's later tokens were written *after reading it*, so it's part of their context. But the model didn't choose those tokens, so we must not train on them: pushing up the log-prob of `232242` inside a tool response would teach the model to *predict* tool results, which is exactly how a model learns to make them up instead of calling the tool. `response_mask = 0` keeps them as context and out of the loss.

### 7.1 Our own token-level tool loop

A harder task makes the tool worth calling: `abc × def − ghij`, which a 0.5B model can't multiply in its head. The calculator is `safe_eval` from §1.1, and its description goes into the prompt through the chat template's `tools=` argument. Qwen's template renders it into the system prompt, along with instructions to write calls as `<tool_call>{json}</tool_call>`:

In [ ]:
CALC_TOOL = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression with numbers, + - * / and parentheses.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "The arithmetic expression, e.g. '123 * 456 - 789'."}},
            "required": ["expression"],
        },
    },
}
TOOL_INSTRUCTION = r"Use the calculate tool for the arithmetic, then give the final answer as \boxed{N}."
HARD_TRAIN, HARD_EVAL = make_problems(2000, seed=3, hard=True), make_problems(100, seed=4, hard=True)


def tool_prompt_ids(question: str) -> list[int]:
    """Like chat_prompt_ids, but the chat template also describes the calculate tool."""
    messages = [{"role": "user", "content": f"{question} {TOOL_INSTRUCTION}"}]
    return tokenizer.apply_chat_template(messages, tools=[CALC_TOOL], add_generation_prompt=True)


print(tokenizer.decode(tool_prompt_ids(HARD_TRAIN[0]["question"])))

**The tool results must be formatted exactly like the chat template formats them.** At inference time (vLLM's chat API, verl's agent loop, or any agent framework), tool results are formatted by the chat template. If our training sequences formatted them even slightly differently, the model would be trained on contexts it never sees in real use. So we build the observation text by hand, and **check it against the template**: render a short conversation that ends in a tool result and a new assistant turn, and verify that it ends with exactly our text.

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)


def run_tool_call(tool_call_json: str) -> str:
    """Parse one model-written tool call and execute it. Errors go back to the model as text."""
    try:
        call = json.loads(tool_call_json)
        arguments = call["arguments"]
        if isinstance(arguments, str):  # some models write the arguments as a JSON string inside the JSON
            arguments = json.loads(arguments)
        if call.get("name") != "calculate":
            return f"Error: unknown tool {call.get('name')!r}"
        value = safe_eval(str(arguments["expression"]))
        if float(value).is_integer():
            return str(int(value))
        return f"{value:.6g}"
    except Exception as exc:
        return f"Error: {exc}"


def observation_text(results: list[str]) -> str:
    """What Qwen's chat template appends after the model's <|im_end|>: the tool results, then a new assistant turn."""
    text = "\n<|im_start|>user"
    for result in results:
        text += f"\n<tool_response>\n{result}\n</tool_response>"
    text += "<|im_end|>\n<|im_start|>assistant\n"
    return text


template_text = tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "What is 2*3?"},
        {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": "calculate", "arguments": {"expression": "2*3"}}}]},
        {"role": "tool", "content": "6"},
    ],
    tools=[CALC_TOOL], add_generation_prompt=True, tokenize=False,
)
print("our observation tokens match the chat template:", template_text.endswith("<|im_end|>" + observation_text(["6"])))
print(repr(observation_text(["6"])))

**`run_episodes`: the multi-turn loop.** Each episode is a dict that grows turn by turn. `response_ids` holds everything after the prompt, the model's turns and the tool outputs, in order, and `mask` says which is which. Here is one episode (the second one printed below), turn by turn:

| Turn | What happens | Appended to `response_ids` | `mask` |
|---|---|---|---|
| 1 | the model writes a tool call and stops | `<tool_call>{"name": "calculate", ...}</tool_call><\|im_end\|>` | 1 |
| | we parse the call and run `safe_eval` → `232242` | `<\|im_start\|>user <tool_response>232242</tool_response><\|im_end\|><\|im_start\|>assistant` | 0 |
| 2 | the model reads the result and answers | `The result ... is \boxed{232242}.<\|im_end\|>` | 1 |
| | no tool call in that turn → the episode is done | | |

At each turn, all still-active episodes are sent to vLLM together, each continuing from its prompt plus everything so far. An episode ends when the model's turn contains **no tool call** (it has answered), when the turn was **cut off** at `max_tokens` (we don't run a half-written tool call), or after `max_turns`. The reward is computed from the model's last turn.

The tool-output tokens get `0.0` as their log-prob, a placeholder: they're masked out, so it's never used.

In [ ]:
def run_episodes(problems, samples_per_prompt: int, max_turns: int = 3, max_tokens: int = 256) -> list[dict]:
    """Multi-turn rollouts: generate, run tool calls, append observations, repeat. All in token space."""
    episodes = [
        {
            "question": problem["question"], "answer": problem["answer"], "group": group,
            "prompt_ids": tool_prompt_ids(problem["question"]),
            "response_ids": [],  # model turns and tool outputs, in order
            "mask": [],          # 1 for model tokens (trained), 0 for tool-output tokens (context only)
            "log_probs": [],     # the engine's log-probs; 0.0 for tool-output tokens
            "tool_calls": 0, "tool_results": [], "done": False,
        }
        for group, problem in enumerate(problems) for _ in range(samples_per_prompt)
    ]
    for turn in range(max_turns):
        active = [episode for episode in episodes if not episode["done"]]
        if not active:
            break
        # One model turn for every active episode, continuing from everything so far
        contexts = [episode["prompt_ids"] + episode["response_ids"] for episode in active]
        turn_outputs = vllm_generate(llm, contexts, 1, max_tokens)
        for episode, output in zip(active, turn_outputs):
            episode["response_ids"] += output["response_ids"]
            episode["mask"] += [1] * len(output["response_ids"])
            episode["log_probs"] += output["log_probs"]
            episode["last_text"] = tokenizer.decode(output["response_ids"])

            calls = TOOL_CALL.findall(episode["last_text"])
            finished = bool(output["response_ids"]) and output["response_ids"][-1] in STOP_IDS  # stopped, not cut off at max_tokens
            if not calls or not finished or turn == max_turns - 1:
                episode["done"] = True
                continue

            # Run the tool calls and append their results as environment tokens (mask 0: not trained)
            results = [run_tool_call(call) for call in calls]
            observation_ids = tokenizer.encode(observation_text(results), add_special_tokens=False)
            episode["response_ids"] += observation_ids
            episode["mask"] += [0] * len(observation_ids)
            episode["log_probs"] += [0.0] * len(observation_ids)
            episode["tool_calls"] += len(calls)
            episode["tool_results"].append(results)
    for episode in episodes:
        episode["reward"] = reward_fn(episode["last_text"], episode["answer"])
    return episodes


def episodes_to_batch(episodes: list[dict]) -> DataProto:
    batch = DataProto.from_dict(
        tensors=make_padded_batch(
            [episode["prompt_ids"] for episode in episodes],
            [episode["response_ids"] for episode in episodes],
            [episode["mask"] for episode in episodes],
        ),
        non_tensors={"uid": np.array([episode["group"] for episode in episodes], dtype=object)},
    )
    response_width = batch.batch["responses"].shape[1]
    token_level_rewards = torch.zeros(len(episodes), response_width)
    rollout_log_probs = torch.zeros(len(episodes), response_width)
    for row, episode in enumerate(episodes):
        token_level_rewards[row, max(len(episode["response_ids"]), 1) - 1] = episode["reward"]  # reward on the last token
        rollout_log_probs[row, : len(episode["log_probs"])] = torch.tensor(episode["log_probs"])
    batch.batch["token_level_rewards"] = token_level_rewards
    batch.batch["rollout_log_probs"] = rollout_log_probs
    return batch


episodes = run_episodes(HARD_TRAIN[:4], samples_per_prompt=4)
for episode in episodes[:2]:
    print("═" * 100, f"\nreward {episode['reward']} | tool calls {episode['tool_calls']} | answer {episode['answer']}")
    print(tokenizer.decode(episode["response_ids"]))

Both episodes called the tool correctly and got the right number back. The first then wrote the answer as plain text (`232,242`) without `\boxed{}`, so it scores 0; the second boxed it and scores 1. The arithmetic is now the tool's job, so the format is almost the only thing left to learn. Below, the same kind of episode is coloured by its mask: green tokens are trained, orange ones are context only.

In [ ]:
import html as _html

from IPython.display import HTML, display


def show_episode(episode: dict) -> None:
    """Green: trained (model) tokens. Orange: environment tokens, context only."""
    # Group consecutive tokens with the same mask value into spans of (mask value, token ids)
    spans = []
    for token_id, trained in zip(episode["response_ids"], episode["mask"]):
        if spans and spans[-1][0] == trained:
            spans[-1][1].append(token_id)
        else:
            spans.append((trained, [token_id]))
    style = {1: "background:#b9f0b4", 0: "background:#ffd59e"}
    body = "".join(
        f'<span style="{style[trained]}">{_html.escape(tokenizer.decode(token_ids)).replace(chr(10), "↵<br>")}</span>'
        for trained, token_ids in spans
    )
    display(HTML(f"<b>reward {episode['reward']}, {sum(episode['mask'])} trained / {len(episode['mask'])} response tokens</b>"
                 f'<div style="font-family:monospace;font-size:12px;border:1px solid #ccc;padding:6px">{body}</div>'))


# Prefer an episode that used the tool and got the answer right
show_episode(max(episodes, key=lambda episode: (episode["tool_calls"] > 0, episode["reward"])))

Training on these episodes needs **no change to the trainer**. The batch has the same keys as before, and `response_mask`, the green/orange colouring above, already zeroes the tool tokens, so the loss ignores them. That's why verl's actor doesn't know or care whether a row came from one turn or ten: the multi-turn logic lives entirely in how the rollout builds the sequence and its mask.

In [ ]:
def tool_rl_step(problems, samples_per_prompt: int = 8) -> tuple[dict, DataProto, list[dict]]:
    timing = {}
    start = time.time()
    episodes = run_episodes(problems, samples_per_prompt)
    timing["gen"] = time.time() - start

    batch = episodes_to_batch(episodes)
    batch.batch["old_log_probs"] = compute_log_probs(policy, batch.batch)
    add_advantages(batch)

    start = time.time()
    stats = ppo_update(
        policy, optimizer, batch.batch,
        old_log_probs=batch.batch["old_log_probs"], advantages=batch.batch["advantages"], loss_fn=verl_ppo_loss,
    )
    timing["update"] = time.time() - start

    start = time.time()
    push_weights(policy, llm)
    timing["sync"] = time.time() - start

    tool_use = float(np.mean([episode["tool_calls"] > 0 for episode in episodes]))  # fraction of episodes that called the tool
    metrics = {**compute_batch_metrics(batch), **stats, "tool_use": tool_use, **{f"time/{name}": seconds for name, seconds in timing.items()}}
    return metrics, batch, episodes


tool_history, all_episodes = [], []
for step in range(6):
    metrics, _, step_episodes = tool_rl_step(step_rng.sample(HARD_TRAIN, 16))
    tool_history.append(metrics)
    all_episodes += step_episodes
    print_step_metrics(step, metrics)
    print(f"        tool use {metrics['tool_use']:.2f}")

The model calls the tool almost every time from the start (`tool use` is 0.95 or more), and the reward jumps from about 0.64 to over 0.9 within two steps. As in §2.5, that jump is the format: the model learns to box the number the tool gave it.

### 7.2 The same thing in verl: `ToolAgentLoop`

verl ships this loop as `ToolAgentLoop` (`verl/experimental/agent_loop/tool_agent_loop.py`): a state machine `PENDING → GENERATING → PROCESSING_TOOLS → GENERATING → …` that works on token IDs, parses tool calls with a `ToolParser` (`hermes` for Qwen), runs tools, and appends their tokens with `response_mask = 0`. Here is its generation step; compare it with `run_episodes`:

In [ ]:
import inspect

from verl.experimental.agent_loop.tool_agent_loop import ToolAgentLoop

src = inspect.getsource(ToolAgentLoop._handle_generating_state).splitlines()
print("\n".join(line for line in src if line.strip() and not line.strip().startswith("#"))[:4000])

| `run_episodes` (§7.1) | verl `ToolAgentLoop` |
|---|---|
| `tool_prompt_ids` (chat template with `tools=`) | `_handle_pending_state` → `apply_chat_template(..., tools=schemas)` |
| `vllm_generate(llm, prompt_ids + response_ids, 1)` | `server_manager.generate(prompt_ids=agent_data.prompt_ids)` |
| `mask += [1] * len(response)` | `agent_data.response_mask += [1] * len(response_ids)` |
| `TOOL_CALL.findall` + `run_tool_call` | `tool_parser.extract_tool_calls` + `_call_tool` |
| `observation_text` → tokens, mask 0 | `_handle_processing_tools_state`: tool messages → template → tokens, mask 0 |
| `max_turns` | `multi_turn.max_assistant_turns` |
| `safe_eval` as the tool | a function decorated with verl's `@function_tool`, or a `BaseTool` subclass |

To use it, we express the agent **in verl's terms**: tools as `@function_tool` functions or `BaseTool` classes, the conversation as a verl dataset row, limits as verl config. For an agent we design for RL from the start, that's a fine deal, and §9.1 runs exactly this with plain verl.

### 7.3 A real agent

Now take an agent that already exists. Calc-X's agent (`examples/calc_x/calc_agent.py`) is ordinary application code:

In [ ]:
print((REPO / "examples/calc_x/calc_agent.py").read_text())

Try to put this inside `run_episodes`, and every assumption breaks:

* **It speaks HTTP, not tokens.** It calls an OpenAI-compatible `/chat/completions` endpoint and gets text and parsed `tool_calls` back. There is no `generate(prompt_ids)` to call.
* **It owns the control flow.** AutoGen decides when to call the model, how to format tool results, and when to stop. With `reflect_on_tool_use=True` its last call even uses a different system prompt (no tools).
* **Its tools are separate processes.** The calculator is an MCP server it starts over stdio. Other agents run tests in Docker, browse the web, or call internal services.
* **It computes its own reward** and reports it at the end.

We could rewrite it as a verl agent loop, but then we'd be training *the rewrite*, and every future change to the real agent would have to be ported. The alternative is to leave it alone and record what it does. But from where? Its logs hold **text**, and training needs **token IDs and log-probs**. Turning logged text back into tokens *usually* works, and that's the trap: when it doesn't, nothing tells us.

**Why text → tokens isn't reliable.** The same text can be split into tokens in more than one way. The tokenizer always produces one particular split (the *canonical* one), but at temperature 1 the model can sample a different one:

In [ ]:
canonical = tokenizer.encode(" calculation", add_special_tokens=False)  # what the tokenizer produces
sampled = tokenizer.encode(" calc", add_special_tokens=False) + tokenizer.encode("ulation", add_special_tokens=False)  # a split the model could sample

print("canonical:", canonical, [tokenizer.decode([t]) for t in canonical])
print("sampled:  ", sampled, [tokenizer.decode([t]) for t in sampled])
print("both decode to the same text:", tokenizer.decode(canonical) == tokenizer.decode(sampled))
print("re-encoding the sampled text gives:", tokenizer.encode(tokenizer.decode(sampled), add_special_tokens=False))

If the model sampled ` calc` + `ulation`, a log of the text says " calculation", and re-encoding it gives the single token 21937, which the model never produced. Training on it would push up a token the model didn't choose, with a log-prob that doesn't belong to it.

Here are two checks on the episodes we just generated, where we *do* know the true tokens:

1. **Text round-trip**: decode each model turn and re-encode it, which catches exactly the problem above.
2. **Re-rendering from an OpenAI-style log**: rebuild the conversation as messages with parsed `tool_calls` (what an HTTP log contains), apply the chat template, and compare with the real token sequence. The template re-serialises the tool-call JSON its own way, and can format whitespace differently from what the model wrote.

In [ ]:
def split_model_turns(episode: dict) -> list[list[int]]:
    """Split an episode's response into the model's turns, dropping the tool outputs between them."""
    turns, current = [], []
    for token_id, trained in zip(episode["response_ids"], episode["mask"]):
        if trained:
            current.append(token_id)
        elif current:  # a tool output starts: the model's turn is over
            turns.append(current)
            current = []
    return turns + ([current] if current else [])


def rerender_from_messages(episode: dict) -> list[int] | None:
    """Rebuild the episode the way an HTTP log would describe it, and re-apply the chat template."""
    messages = [{"role": "user", "content": f"{episode['question']} {TOOL_INSTRUCTION}"}]
    turns = split_model_turns(episode)
    for i, turn in enumerate(turns):
        text = tokenizer.decode(turn).removesuffix("<|im_end|>")
        calls = TOOL_CALL.findall(text)
        if i < len(turns) - 1 and calls:
            try:
                parsed = [json.loads(call) for call in calls]
                tool_calls = [{"type": "function", "function": {"name": call["name"], "arguments": call["arguments"]}} for call in parsed]
            except (json.JSONDecodeError, KeyError, TypeError):
                return None
            messages.append({"role": "assistant", "content": text.split("<tool_call>")[0].strip(), "tool_calls": tool_calls})
            messages += [{"role": "tool", "content": result} for result in episode["tool_results"][i]]
        else:
            messages.append({"role": "assistant", "content": text})
    return tokenizer.apply_chat_template(messages, tools=[CALC_TOOL])


# 1. Does decoding a model turn to text and re-encoding it give back the same tokens?
turns = [turn for episode in all_episodes for turn in split_model_turns(episode)]
roundtrip_bad = []
for turn in turns:
    reencoded = tokenizer.encode(tokenizer.decode(turn), add_special_tokens=False)
    if reencoded != turn:
        roundtrip_bad.append(turn)
print(f"1. text round-trip: {len(roundtrip_bad)} of {len(turns)} model turns ({len(roundtrip_bad) / max(len(turns), 1):.1%}) re-encode to different tokens")

# 2. Does re-rendering a whole episode from messages give back the same tokens?
with_tools = [episode for episode in all_episodes if episode["tool_calls"] > 0]
matches = []  # True: same tokens, False: different tokens, None: tool calls didn't parse
for episode in with_tools:
    actual = episode["prompt_ids"] + episode["response_ids"]
    rebuilt = rerender_from_messages(episode)
    matches.append(None if rebuilt is None else rebuilt[: len(actual)] == actual)
n_ok = sum(match is True for match in matches)
print(f"2. re-rendered tool-using episodes: {n_ok} of {len(matches)} reproduce the exact tokens "
      f"({sum(match is None for match in matches)} had tool calls that don't even parse as JSON)")
example = next((episode for episode, match in zip(with_tools, matches) if match is False), None)
if example is not None:
    actual, rebuilt = example["prompt_ids"] + example["response_ids"], rerender_from_messages(example)
    i = 0
    while actual[i] == rebuilt[i]:  # walk forward to the first token where they differ
        i += 1
    print(f"\nfirst difference at token {i}:\n  model produced: {tokenizer.decode(actual[i - 8: i + 12])!r}\n  log re-renders: {tokenizer.decode(rebuilt[i - 8: i + 12])!r}")

With this simple agent and a model that follows its template closely, mismatches are rare: under 1% of episodes. The example printed above is a typical one; in the run saved here, the model ended its answer with `<|im_end|>`, and the template added a newline after it that the model never wrote. Every such mismatch is a training example where the loss pushes on tokens the model never chose, and nothing flags it. With messier agents (frameworks that reformat messages, models that don't follow the template exactly, long multi-turn traces) the rate goes up, and a real agent's logs often don't contain the full prompt the model saw at all.

So there's exactly one place where a real agent's training data can be captured correctly: **at the HTTP boundary between the agent and the inference engine**, where the engine still knows the exact token IDs and log-probs. Let's build that.

## 8. Swap 6: recording a real agent at the HTTP boundary

### 8.1 Our own gateway

The idea in one sentence: give each rollout its **own base URL**, and put a proxy there that forwards the agent's requests to vLLM, asks vLLM to also return token IDs and log-probs, and records them under that rollout's ID.

```
agent (rollout r42)                       our gateway                              vLLM server
  POST /rollout/r42/v1/chat/completions ──►  forward, plus "return token ids     ──►  /v1/chat/completions
                                             and log-probs"
  ◄── normal OpenAI response ───────────────  record under "r42":                 ◄──  response + token ids
      (text + tool_calls)                     prompt_ids, response_ids, log_probs       + log-probs

  POST /rollout/r42/events {"reward": 1} ──►  RECORDS["r42"]["reward"] = 1.0
```

Why a URL per rollout? We can't change the agent's code to tag its requests with a rollout ID, but every agent framework lets us configure the model's base URL. Putting the rollout ID *in* the URL tags every request the agent makes, for free. The agent needs no changes except its base URL.

We need vLLM as an HTTP server now. Two flags matter for RL:

* `--generation-config vllm` ignores the model's chat sampling defaults from §2.3 (top-k 20, top-p 0.8, repetition penalty 1.1). Without it, every request that doesn't set them explicitly samples from the wrong distribution.
* `VLLM_SERVER_DEV_MODE=1` exposes `/collective_rpc` and `/reset_prefix_cache`, the HTTP versions of the weight-sync calls from §2.4. They're development endpoints, so the server listens on 127.0.0.1 only.

Calc-X questions are harder than our arithmetic, and the 0.5B model almost never solves them, which would leave every GRPO group with zero advantage (§1.5). From here on we use the 1.5B model the Calc-X recipe trains, so some attempts succeed.

First, free the GPU: the in-process vLLM engine and the policy from §7 go away.

In [ ]:
llm.llm_engine.engine_core.shutdown()
del llm, policy, optimizer
gc.collect()
torch.cuda.empty_cache()
time.sleep(5)
print(f"GPU free: {gpu_free_gib():.0f} GiB")

CALC_X_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # same tokenizer as the 0.5B model
VLLM_PORT = 8110
VLLM_ROOT = f"http://127.0.0.1:{VLLM_PORT}"
start_service("vllm", [
    str(VENV_BIN / "vllm"), "serve", CALC_X_MODEL, "--host", "127.0.0.1", "--port", str(VLLM_PORT),
    "--gpu-memory-utilization", "0.3", "--max-model-len", "8192", "--seed", str(SEED),
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes", "--generation-config", "vllm",
], env={"VLLM_SERVER_DEV_MODE": "1"})
wait_until_healthy("vllm", f"{VLLM_ROOT}/health")

The whole gateway is two routes. One proxies chat completions and records tokens; the other accepts the agent's reward event. Before forwarding a request, it rewrites a few fields:

| Field | Set to | Why |
|---|---|---|
| `model` | the policy's name | the agent asks for `"auto"`; vLLM needs the real name |
| `temperature`, `top_p` | 1.0 | sample from the policy itself (§1.2) |
| `return_token_ids` | `True` | vLLM adds `prompt_token_ids` and `token_ids` to the response |
| `logprobs` | `True` | vLLM adds the log-prob of every sampled token |
| `stream` | `False` | one complete response, so we can record it in one piece |

The agent gets back a normal OpenAI response (the extra fields are just ignored), so it never knows it's being recorded.

In [ ]:
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

RECORDS: dict[str, dict] = {}
gateway = FastAPI()


def get_record(rollout_id: str) -> dict:
    return RECORDS.setdefault(rollout_id, {"calls": [], "reward": None})


@gateway.get("/healthz")
async def healthz():
    return {"ok": True}


@gateway.post("/rollout/{rollout_id}/v1/chat/completions")
async def chat_completions(rollout_id: str, request: Request):
    body = await request.json()
    body.update(model=CALC_X_MODEL, temperature=1.0, top_p=1.0, logprobs=True, return_token_ids=True, stream=False)
    async with httpx.AsyncClient(timeout=300) as client:
        upstream = await client.post(f"{VLLM_ROOT}/v1/chat/completions", json=body)
    data = upstream.json()
    if upstream.status_code == 200:
        choice = data["choices"][0]
        get_record(rollout_id)["calls"].append({
            "prompt_ids": data.get("prompt_token_ids") or choice.get("prompt_token_ids"),
            "response_ids": choice["token_ids"],
            "log_probs": [t["logprob"] for t in choice["logprobs"]["content"]],
        })
    return JSONResponse(data, status_code=upstream.status_code)


@gateway.post("/rollout/{rollout_id}/events")
async def events(rollout_id: str, request: Request):
    event = await request.json()
    if event.get("event_type") == "reward":
        get_record(rollout_id)["reward"] = float(event["data"]["value"])
    return {"ok": True}


GATEWAY_PORT = 8291
GATEWAY_URL = f"http://127.0.0.1:{GATEWAY_PORT}"
gateway_server = uvicorn.Server(uvicorn.Config(gateway, host="127.0.0.1", port=GATEWAY_PORT, log_level="warning"))
threading.Thread(target=gateway_server.run, daemon=True).start()
while not gateway_server.started:
    time.sleep(0.1)
print("gateway up:", httpx.get(f"{GATEWAY_URL}/healthz").json())

And a "controller": start one agent process per rollout with the environment variables `calc_agent.py` reads, and wait for all of them. The agent posts its reward to `AGL_EVENT_URL` and talks to the model through `AGL_OPENAI_BASE_URL`, and both point at our gateway.

In [ ]:
import uuid

(WORK / "agents").mkdir(exist_ok=True)


def launch_agent(rollout_id: str, question: str, result) -> subprocess.Popen:
    env = {
        **os.environ, "QUESTION": question, "RESULT": str(result), "AGL_KEY": "not-checked-by-our-gateway",
        "AGL_OPENAI_BASE_URL": f"{GATEWAY_URL}/rollout/{rollout_id}/v1",
        "AGL_EVENT_URL": f"{GATEWAY_URL}/rollout/{rollout_id}/events",
    }
    with open(WORK / "agents" / f"{rollout_id}.log", "w") as log:
        return subprocess.Popen(
            [sys.executable, "-c", "import asyncio, calc_agent; asyncio.run(calc_agent.Agent().run())"],
            cwd=REPO / "examples/calc_x", env=env, stdout=log, stderr=subprocess.STDOUT,
        )


def run_agents(jobs: list[tuple[str, str, object]], timeout: float = 300) -> dict[str, int]:
    """jobs: (rollout_id, question, result). Runs one agent process per job, all at once; returns their exit codes."""
    procs = {rollout_id: launch_agent(rollout_id, question, result) for rollout_id, question, result in jobs}
    deadline = time.time() + timeout
    for proc in procs.values():
        try:
            proc.wait(max(1.0, deadline - time.time()))
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    return {rollout_id: proc.returncode for rollout_id, proc in procs.items()}


calc_train = pd.read_parquet(CALC_X_DATA / "train.parquet")
demo_row = calc_train.iloc[0]
rollout_id = f"demo-{uuid.uuid4().hex[:6]}"
print("exit code:", run_agents([(rollout_id, demo_row["question"], demo_row["result"])])[rollout_id])
recorded = RECORDS[rollout_id]
print(f"question: {demo_row['question']}\nreward: {recorded['reward']} (truth {demo_row['result']!r})")
for i, call in enumerate(recorded["calls"]):
    print(f"call {i}: {len(call['prompt_ids'])} prompt tokens, {len(call['response_ids'])} response tokens, {len(call['log_probs'])} log-probs")
    print("   ", repr(tokenizer.decode(call["response_ids"])[:200]))

The unmodified AutoGen agent ran, and we hold **exactly** the tokens vLLM sampled for every one of its model calls, with their log-probs, and its reward. That's enough to train on.

### 8.2 Training through our gateway

The batch is built **per model call** (the "transition" level): each call is one row, with its own prompt (whatever the agent sent) and response. That's robust to anything the agent does to its context between calls, like the system-prompt change from `reflect_on_tool_use`. Two consequences:

* All rows of a rollout share the rollout's reward.
* The GRPO advantage must be computed **per rollout**, not per row. Otherwise a rollout that made 3 calls would count 3 times in its group's mean and std. Here's the difference for two rollouts of one question, where A succeeded using 3 model calls and B failed using 1:

In [ ]:
# Wrong: treat every model call as its own sample. A's 3 calls count 3 times
per_row = grpo_advantages(torch.tensor([1.0, 1.0, 1.0, 0.0]), [0, 0, 0, 0])

# Right: one advantage per rollout, then copy it onto that rollout's rows
per_rollout = grpo_advantages(torch.tensor([1.0, 0.0]), [0, 0])
copied_to_rows = per_rollout[[0, 0, 0, 1]]  # A's three calls, then B's one

print("per row (wrong):              ", [round(a, 2) for a in per_row.tolist()])
print("per rollout, copied to rows:  ", [round(a, 2) for a in copied_to_rows.tolist()])

Computed per row, A's three calls pull the group mean up to 0.75, so A's success looks "expected" and gets only +0.5, while B's failure gets −1.5. How many calls an agent happened to make would decide the training signal. Computed per rollout, A and B are one success and one failure: ±0.71, as they should be.

Training then uses the same functions as before, and weight sync goes over HTTP.

In [ ]:
def push_weights_http(model) -> None:
    path = save_for_inference(model)
    httpx.post(f"{VLLM_ROOT}/collective_rpc", json={"method": "reload_weights", "kwargs": {"weights_path": path}}, timeout=600).raise_for_status()
    httpx.post(f"{VLLM_ROOT}/reset_prefix_cache", timeout=60).raise_for_status()


def gateway_batch(rollouts: list[tuple[str, int]]) -> tuple[DataProto, dict]:
    """rollouts: (rollout_id, question_index). One row per recorded model call; advantages per rollout."""
    # One reward per rollout, and GRPO advantages across the rollouts of each question
    rewards = torch.tensor([RECORDS.get(rollout_id, {}).get("reward") or 0.0 for rollout_id, _ in rollouts])
    rollout_advantages = grpo_advantages(rewards, [question for _, question in rollouts])

    # One training row per model call; every call in a rollout shares that rollout's reward and advantage
    prompts, responses, engine_log_probs, row_advantages, row_rewards, uids = [], [], [], [], [], []
    for (rollout_id, question), advantage, reward in zip(rollouts, rollout_advantages.tolist(), rewards.tolist()):
        for call in RECORDS.get(rollout_id, {}).get("calls", []):
            prompts.append(call["prompt_ids"])
            responses.append(call["response_ids"])
            engine_log_probs.append(call["log_probs"])
            row_advantages.append(advantage)
            row_rewards.append(reward)
            uids.append(question)

    batch = DataProto.from_dict(tensors=make_padded_batch(prompts, responses), non_tensors={"uid": np.array(uids, dtype=object)})
    mask = batch.batch["response_mask"]
    batch.batch["advantages"] = torch.tensor(row_advantages)[:, None] * mask
    token_level_rewards, rollout_log_probs = torch.zeros(mask.shape), torch.zeros(mask.shape)
    for row, (response, log_probs) in enumerate(zip(responses, engine_log_probs)):
        token_level_rewards[row, max(len(response), 1) - 1] = row_rewards[row]
        rollout_log_probs[row, : len(log_probs)] = torch.tensor(log_probs)
    batch.batch["token_level_rewards"] = token_level_rewards
    batch.batch["rollout_log_probs"] = rollout_log_probs

    info = {
        "rollout_reward": rewards.mean().item(),
        "rows": len(batch),
        "rollouts_without_calls": sum(not RECORDS.get(rollout_id, {}).get("calls") for rollout_id, _ in rollouts),
    }
    return batch, info


policy = load_policy(CALC_X_MODEL)
optimizer = make_optimizer(policy, LR)
calc_rng = random.Random(SEED)
gateway_history, gateway_jobs = [], []
for step in range(3):
    questions = calc_train.iloc[calc_rng.sample(range(len(calc_train)), 8)]
    rollouts = []  # (rollout_id, question index)
    jobs = []      # (rollout_id, question, reference result), what run_agents needs
    for q_index in range(len(questions)):
        question = questions.iloc[q_index]
        for _ in range(4):  # 4 rollouts per question, each with its own id
            rollout_id = f"step{step}-q{q_index}-{uuid.uuid4().hex[:6]}"
            rollouts.append((rollout_id, q_index))
            jobs.append((rollout_id, question["question"], question["result"]))
    gateway_jobs += jobs  # kept for the LLM judge in §8.4

    start = time.time()
    run_agents(jobs)
    agent_seconds = time.time() - start

    batch, info = gateway_batch(rollouts)
    batch.batch["old_log_probs"] = compute_log_probs(policy, batch.batch)
    stats = ppo_update(
        policy, optimizer, batch.batch,
        old_log_probs=batch.batch["old_log_probs"], advantages=batch.batch["advantages"],
        mini_batch_size=len(batch),  # one optimizer step per training step
        loss_fn=verl_ppo_loss,
    )
    push_weights_http(policy)

    metrics = {**compute_batch_metrics(batch), **stats, "time/gen": agent_seconds}
    gateway_history.append({**metrics, **info})
    print(f"step {step} | rollout reward {info['rollout_reward']:.3f} | {info['rows']} rows from {len(rollouts)} rollouts "
          f"({info['rollouts_without_calls']} without model calls) | logp mismatch {metrics['logp_mismatch']:.4f} | grad_norm {metrics['grad_norm']:.3f} | agents {agent_seconds:.0f}s")

**A real, unmodified agent is now training**, through about 60 lines of gateway and 20 lines of process launching. We shouldn't read much into the rewards: at temperature 1 this model rarely gets a Calc-X question fully right, and three small steps only nudge that. What matters is that every number in the step line came from tokens the agent's own calls produced. That's the core idea of Agent Lightning, and we now know it from the inside.

### 8.3 The swap: Agent Lightning's Gateway and Controller

Our toy gets the idea right but cuts every corner: rollouts live in a Python dict, agents are launched with a `Popen` loop, a crashed agent silently becomes "reward 0", and every model call becomes its own training row. Agent Lightning's **Gateway** (a rollout store plus the same kind of token-recording proxy) and **Controller** (which launches one agent process, or Kubernetes Job, per queued rollout) replace those two pieces. As with every other swap, let's run the real thing on the same job, with the same agent, the same questions and the same vLLM server, and check what changes.

`AglRolloutManager.enqueue_and_wait_until_completed` is the exact call Agent Lightning's trainer makes at every step: register the vLLM endpoint, queue `n` rollouts per question, and wait for the Controller to run them.

In [ ]:
from agentlightning.verl.agl_rollout_manager import AglRolloutManager

AGL_PORT, AGL_KEY = 8283, "rl-from-scratch"
AGL_URL = f"http://127.0.0.1:{AGL_PORT}"
start_service("agl_gateway", [
    sys.executable, "-m", "agentlightning.server", "host=127.0.0.1", f"port={AGL_PORT}", f"key={AGL_KEY}",
    f"default_proxy.model_name={CALC_X_MODEL}", f"hydra.run.dir={WORK}/hydra/gateway", "hydra.output_subdir=null",
])
wait_until_healthy("agl_gateway", f"{AGL_URL}/healthz")
start_service("agl_controller", [
    sys.executable, "-m", "agentlightning.controller", "runner_type=local", f"agl_server.url={AGL_URL}", f"agl_server.key={AGL_KEY}",
    "local_runner.poll_interval=1", "local_runner.maximum_size=128", f"hydra.run.dir={WORK}/hydra/controller", "hydra.output_subdir=null",
], cwd=REPO / "examples/calc_x")
time.sleep(3)
assert SERVICES["agl_controller"].poll() is None, log_tail("agl_controller")

manager = AglRolloutManager(
    agl_base_url=AGL_URL, agl_key=AGL_KEY, model=CALC_X_MODEL, step=0, train_rollout_n=4,
    rollout_timeout_seconds=300, poll_interval_seconds=2,
    local_agent_class="examples.calc_x.calc_agent.Agent",
    local_env_map={"QUESTION": "input.question", "RESULT": "input.result"},  # how the Controller fills the env vars our launch_agent set
)
manager.delete_model()
manager.register_model([f"{VLLM_ROOT}/v1"])

t0 = time.time()
completed = manager.enqueue_and_wait_until_completed({column: np.array(questions[column].tolist(), dtype=object) for column in questions.columns}, is_train=True)
print(f"\n{len(completed)} rollouts in {time.time() - t0:.0f}s, states: {pd.Series([str(r.rollout_state) for r in completed]).value_counts().to_dict()}")
print(f"mean reward {np.mean([r.final_reward or 0.0 for r in completed]):.3f} (our gateway's last step: {gateway_history[-1]['rollout_reward']:.3f})")

Each `CompletedRollout` holds one **triplet** per model call (prompt token IDs, response token IDs and vLLM's log-probs) plus the final reward: the same information as our `RECORDS` dict. Agent Lightning's `RolloutAdapter` then turns them into a `DataProto` with verl's keys, which is the job our `gateway_batch` did.

Let's build that batch one row per model call, as ours did, and check the two things our toy relied on:

1. the recorded log-probs agree with the trainer's, to the same small mismatch as before;
2. the rollout-level advantages, computed by Agent Lightning's `compute_rollout_level_advantage`, match ours.

One difference shows up immediately: the adapter pads every row to fixed maximum lengths (2048 prompt + 8192 response tokens, Agent Lightning's defaults), so every batch has the same shape no matter what the agent did. Our `make_padded_batch` pads to the longest row instead. Neither wastes much compute in practice, because verl strips the padding before the forward pass (`use_remove_padding=True`); we do the same by hand here.

In [ ]:
from agentlightning.verl.rollout_adapter import RolloutAdapter
from agentlightning.verl.rollout_level_advantage import compute_rollout_level_advantage
from verl.trainer.ppo.ray_trainer import compute_response_mask


def make_adapter(level: str) -> RolloutAdapter:
    return RolloutAdapter(max_prompt_length=2048, max_response_length=8192, device=torch.device("cpu"), pad_token_id=PAD_ID,
                          reward_fillna_value=0.0, trace_aggregator_level=level, tokenizer=tokenizer)


def unpad_batch(batch: DataProto):
    """Strip fixed-length padding, back to per-row token lists."""
    prompt_width = batch.batch["prompts"].shape[1]
    attention_mask = batch.batch["attention_mask"]
    response_lens = attention_mask[:, prompt_width:].sum(-1).tolist()  # real tokens in each row's response
    prompts, responses, masks, log_probs = [], [], [], []
    for row, length in enumerate(response_lens):
        is_real_prompt_token = attention_mask[row, :prompt_width].bool()
        prompts.append(batch.batch["prompts"][row][is_real_prompt_token].tolist())
        responses.append(batch.batch["responses"][row, :length].tolist())
        masks.append(batch.batch["response_mask"][row, :length].tolist())
        log_probs.append(batch.batch["rollout_log_probs"][row, :length].float())
    return prompts, responses, masks, log_probs


agl_batch, _ = make_adapter("transition").get_train_data_batch(completed)
if "response_mask" not in agl_batch.batch.keys():  # one row per call: every response token is trained, so the trainer derives the mask
    agl_batch.batch["response_mask"] = compute_response_mask(agl_batch)
prompts, responses, masks, engine_log_probs_per_row = unpad_batch(agl_batch)
our_batch = make_padded_batch(prompts, responses, masks)
print(f"{len(agl_batch)} rows; Agent Lightning's padding: prompts {tuple(agl_batch.batch['prompts'].shape)}, responses {tuple(agl_batch.batch['responses'].shape)}"
      f" | ours: prompts {tuple(our_batch['prompts'].shape)}, responses {tuple(our_batch['responses'].shape)}")

# 1. Same tokens, same log-probs?
trainer_log_probs = compute_log_probs(policy, our_batch)
engine_log_probs = torch.zeros_like(trainer_log_probs)
for row, log_probs in enumerate(engine_log_probs_per_row):
    engine_log_probs[row, : len(log_probs)] = log_probs
trained = our_batch["response_mask"].bool()
print(f"1. logp mismatch, trainer vs vLLM: {(trainer_log_probs - engine_log_probs)[trained].abs().mean():.4f} through Agent Lightning's Gateway"
      f" | {gateway_history[-1]['logp_mismatch']:.4f} through ours (last step)")

# 2. Same advantages? Agent Lightning's, per row, vs ours computed per rollout
agl_batch.non_tensor_batch["uid"] = agl_batch.non_tensor_batch["data_id_list"]
agl_batch.batch["token_level_rewards"] = agl_batch.batch["token_level_scores"]
agl_batch, _ = compute_rollout_level_advantage(agl_batch, adv_estimator="grpo", gamma=1.0, lam=1.0, num_repeat=4, norm_adv_by_std_in_grpo=True)
row_rollouts = agl_batch.non_tensor_batch["rollout_id_list"]  # which rollout each row came from
completed_by_id = {rollout.rollout_id: rollout for rollout in completed}
rollout_ids = list(dict.fromkeys(row_rollouts))  # unique, in order
our_rollout_advantages = grpo_advantages(
    torch.tensor([completed_by_id[rollout_id].final_reward or 0.0 for rollout_id in rollout_ids]),
    [completed_by_id[rollout_id].data_id for rollout_id in rollout_ids],  # group = question
)
our_rollout_advantages = dict(zip(rollout_ids, our_rollout_advantages.tolist()))
# Agent Lightning puts the advantage on every token of a row; average over the row to read it back
agl_mask = agl_batch.batch["response_mask"].float()
agl_advantage_sums = (agl_batch.batch["advantages"].float() * agl_mask).sum(-1)
agl_row_advantages = (agl_advantage_sums / agl_mask.sum(-1).clamp(min=1)).tolist()
print("2. rollout-level advantages match ours:",
      np.allclose(agl_row_advantages, [our_rollout_advantages[rollout_id] for rollout_id in row_rollouts], atol=1e-2),
      f"({sum(advantage != 0 for advantage in our_rollout_advantages.values())} of {len(our_rollout_advantages)} rollouts with non-zero advantage)")

Same data, same maths. So what does the real one add? Mostly bookkeeping that only matters at scale: a rollout store with explicit states, so a crashed agent is recorded as **failed** and left out of the batch instead of being trained on as a "reward 0" example; a Controller with concurrency limits and timeouts that can launch Kubernetes Jobs instead of local processes; and a model registry, so the trainer can point the proxy at whichever vLLM replicas hold the latest weights, and pause it during weight sync.

One addition is worth looking at here, because it changes the batch. Our gateway made every model call its own row. But a typical agent's next call *extends* its last one: the new prompt is the old prompt, plus the model's response, plus the tool output. Then the two calls are really **one growing token sequence**, and can be trained as a single row, just like the §7 episodes:

```
call 1:      prompt P1                         → response R1
call 2:      prompt P1 + R1 + tool output T1   → response R2

merged row:  prompt P1 │ R1 (mask 1)  T1 (mask 0)  R2 (mask 1)
```

The merged row processes P1 once instead of twice, and the model's tokens are trained in their full context. That's the `trajectory` level, Agent Lightning's default. It merges calls whenever this **prefix continuity** holds (call *k+1*'s prompt starts with exactly call *k*'s prompt + response), and starts a new row when it doesn't. Let's see what happens with the Calc-X agent:

In [ ]:
def find_prefix_break(rollout):
    """Where call k+1's prompt stops extending call k's prompt + response, if it ever does."""
    for call, next_call in zip(rollout.triplets, rollout.triplets[1:]):
        context = call.prompt["token_ids"] + call.response["token_ids"]  # everything the model had seen and said so far
        next_prompt = next_call.prompt["token_ids"]
        if next_prompt[: len(context)] != context:
            first_diff = 0  # walk forward to the first token where they differ
            while first_diff < min(len(context), len(next_prompt)) and context[first_diff] == next_prompt[first_diff]:
                first_diff += 1
            return context, next_prompt, first_diff
    return None


traj_batch, traj_metrics = make_adapter("trajectory").get_train_data_batch(completed)
multi_call = [rollout for rollout in completed if len(rollout.triplets or []) > 1]
print(f"{len(completed)} rollouts, {sum(len(rollout.triplets or []) for rollout in completed)} model calls; {len(multi_call)} rollouts made more than one call")
print(f"  transition level: {len(agl_batch)} rows")
print(f"  trajectory level: {len(traj_batch)} rows ({traj_metrics['training/n_unmerged_rollouts']} rollouts couldn't be merged)")
print(f"  rollouts with a prefix-continuity break: {sum(find_prefix_break(rollout) is not None for rollout in multi_call)} of {len(multi_call)}")

example = next((found for found in map(find_prefix_break, multi_call) if found is not None), None)
if example is not None:
    context, next_prompt, i = example
    window = slice(max(0, i - 25), i + 25)
    print(f"\nfirst difference at token {i}:\n  call k prompt + response: …{tokenizer.decode(context[window])!r}\n  call k+1 prompt:          …{tokenizer.decode(next_prompt[window])!r}")

Every rollout that made more than one call breaks continuity, and always at the same place: token 26, inside the system prompt. The Calc-X agent uses AutoGen's `reflect_on_tool_use=True`, so its last call (the "reflection" that writes the final answer) is made *without* tools, and the whole `# Tools` block disappears from the system prompt. From that token on, the second call's prompt is a different sequence, so it can't extend the first one, and the trajectory level falls back to one row per call: the same rows as the transition level.

Nothing is wrong with the training data; it's just less efficient: every row re-processes its whole prompt, including everything the earlier calls already processed. This is a practical lesson the toy couldn't teach us: **an agent's code decides how efficiently it can be trained**, often through framework settings that look unrelated to training.

For everything else (the rollout store's API, the Controller's settings, asynchronous training, Kubernetes), Agent Lightning's [documentation](https://microsoft.github.io/agent-lightning/) is the place to go. The policy, our vLLM server and Agent Lightning's services have done their job, so let's free the GPU. (The LLM judges that read these same rollouts are [notebook 4](4_rewards_and_judges.ipynb).)

In [ ]:
gateway_server.should_exit = True
for name in ["agl_controller", "agl_gateway", "vllm"]:
    stop_service(name)
del policy, optimizer
gc.collect()
torch.cuda.empty_cache()
time.sleep(5)
print(f"GPU free: {gpu_free_gib():.0f} GiB")

## 9. The real thing

### 9.1 Plain verl: the §7 tool task, with verl's `ToolAgentLoop`

Everything from §1–§7.2 is now verl's job: Ray worker groups, FSDP actor, colocated vLLM with weight sync, `DataProto`, GRPO, PPO-clip, and the multi-turn tool loop. What *we* provide is the part that is specific to our task, in verl's format:

* **the tool**, as a function decorated with `@function_tool` (its docstring and type hints become the JSON schema in the prompt);
* **the reward**, as `compute_score(data_source, solution_str, ground_truth, extra_info)`;
* **the data**, as parquet rows with a chat `prompt`, a `data_source` and `reward_model.ground_truth`.

All three reuse `arith_task`.

In [ ]:
%%writefile tutorial/generated/verl_calc_tool.py
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "lib"))
from arith_task import safe_eval  # noqa: E402
from verl.tools.function_tool import function_tool  # noqa: E402


@function_tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression with numbers, + - * / and parentheses.

    Args:
        expression: The arithmetic expression, e.g. '123 * 456 - 789'.
    """
    try:
        value = safe_eval(expression)
    except Exception as exc:
        return f"Error: {exc}"
    return str(int(value)) if float(value).is_integer() else f"{value:.6g}"

In [ ]:
%%writefile tutorial/generated/verl_reward.py
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "lib"))
from arith_task import reward_fn  # noqa: E402


def compute_score(data_source, solution_str, ground_truth, extra_info=None, **kwargs):
    return reward_fn(solution_str, int(ground_truth))

In [ ]:
import datasets


def to_verl_rows(problems):
    return [
        {"data_source": "arith", "prompt": [{"role": "user", "content": f"{problem['question']} {TOOL_INSTRUCTION}"}],
         "reward_model": {"style": "rule", "ground_truth": str(problem["answer"])}, "extra_info": {"index": index}}
        for index, problem in enumerate(problems)
    ]


datasets.Dataset.from_list(to_verl_rows(HARD_TRAIN[:512])).to_parquet(str(WORK / "arith_train.parquet"))
datasets.Dataset.from_list(to_verl_rows(HARD_EVAL[:64])).to_parquet(str(WORK / "arith_val.parquet"))

PLAIN_VERL_METRICS = WORK / "plain_verl_metrics.jsonl"
PLAIN_VERL_METRICS.unlink(missing_ok=True)
verl_cmd = [
    sys.executable, "-m", "verl.trainer.main_ppo",
    "algorithm.adv_estimator=grpo",
    f"data.train_files={WORK / 'arith_train.parquet'}", f"data.val_files={WORK / 'arith_val.parquet'}",
    "data.train_batch_size=16", "data.max_prompt_length=1024", "data.max_response_length=1024", "data.dataloader_num_workers=0",
    f"actor_rollout_ref.model.path={MODEL}",
    "actor_rollout_ref.model.use_remove_padding=True", "actor_rollout_ref.model.enable_gradient_checkpointing=True",
    f"actor_rollout_ref.actor.optim.lr={LR}",
    "actor_rollout_ref.actor.ppo_mini_batch_size=16", "actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=16",
    "actor_rollout_ref.actor.use_kl_loss=False", "actor_rollout_ref.actor.entropy_coeff=0",
    "actor_rollout_ref.actor.clip_ratio_low=0.2", "actor_rollout_ref.actor.clip_ratio_high=0.28",
    "actor_rollout_ref.rollout.name=vllm", "actor_rollout_ref.rollout.tensor_model_parallel_size=1",
    "actor_rollout_ref.rollout.gpu_memory_utilization=0.4", "actor_rollout_ref.rollout.n=8",
    "actor_rollout_ref.rollout.log_prob_micro_batch_size_per_gpu=16",
    "actor_rollout_ref.rollout.multi_turn.enable=True", "actor_rollout_ref.rollout.multi_turn.format=hermes",
    "actor_rollout_ref.rollout.multi_turn.max_assistant_turns=3",
    f"actor_rollout_ref.rollout.multi_turn.function_tool_path={GENERATED / 'verl_calc_tool.py'}",
    "actor_rollout_ref.rollout.agent.default_agent_loop=tool_agent", "actor_rollout_ref.rollout.agent.num_workers=2",
    f"reward.custom_reward_function.path={GENERATED / 'verl_reward.py'}", "reward.custom_reward_function.name=compute_score",
    "reward.num_workers=2",
    "trainer.n_gpus_per_node=1", "trainer.nnodes=1", "trainer.total_training_steps=3",
    "trainer.val_before_train=True", "trainer.test_freq=3", "trainer.save_freq=-1",
    "trainer.logger=[console,file]", "trainer.project_name=rl_training_from_scratch", "trainer.experiment_name=plain_verl",
    f"trainer.rollout_data_dir={WORK / 'verl_rollouts'}", f"trainer.validation_data_dir={WORK / 'verl_val'}",
]
print(" \\\n  ".join(verl_cmd[2:]))

Let's map the config back onto what we built:

| Config | Our code |
|---|---|
| `algorithm.adv_estimator=grpo` | `grpo_advantages` / `compute_grpo_outcome_advantage` (§1.5, §2.2) |
| `actor.clip_ratio_low/high`, `ppo_mini_batch_size`, `ppo_micro_batch_size_per_gpu` | `ppo_clip_loss` and `ppo_update`'s mini/micro-batch loops (§1.7, §2.1) |
| `rollout.name=vllm`, `gpu_memory_utilization`, `n` | `RolloutWorker` and `vllm_generate(samples_per_prompt=...)` (§2.3, §5) |
| `rollout.multi_turn.*`, `agent.default_agent_loop=tool_agent` | `run_episodes` (§7.1) |
| `trainer.n_gpus_per_node`, `nnodes` | the resource pool (§6.2) |
| `reward.custom_reward_function` | `reward_fn` |
| `model.use_remove_padding=True` | (new) instead of padding rows to a rectangle (§2.1), verl packs all the real tokens of a batch into one long sequence, so no compute is spent on padding. `global_seqlen` in the log is that sequence's length |

**Watch the units.** `data.train_batch_size=16` is in **prompts**: each step takes 16 questions, and `rollout.n=8` samples each one 8 times, giving 128 episodes. `ppo_mini_batch_size=16` is also in prompts, and verl multiplies it by `n` too: 16 × 8 = 128 rows, the whole batch. So each training step is **one** optimizer step, every update happens at ratio ρ = 1, and the clip never acts. Keep that in mind when reading the metrics below. (`ppo_micro_batch_size_per_gpu`, by contrast, counts rows.)

This takes a few minutes, mostly start-up (Ray, the FSDP actor, vLLM with CUDA graphs). Selected log lines are printed; the full log is in `/tmp/rl-from-scratch/logs/plain_verl.log`.

In [ ]:
start_service("plain_verl", verl_cmd, cwd=WORK, env={"VERL_FILE_LOGGER_PATH": str(PLAIN_VERL_METRICS), "WANDB_MODE": "disabled"})
exit_code = follow_log("plain_verl", r"step:\d+ |Initial validation|Training Progress")
print(f"\nverl exited with code {exit_code}")
if exit_code != 0:
    raise RuntimeError(log_tail("plain_verl", 60))

In [ ]:
def read_metrics(path: Path, patterns: list[str]) -> pd.DataFrame:
    """verl's file logger writes one JSON line per step; keep the metrics whose names match any pattern, in pattern order."""
    records = [json.loads(line) for line in path.read_text().splitlines()]
    all_names = sorted({name for record in records for name in record["data"]})
    keep = [name for pattern in patterns for name in all_names if re.fullmatch(pattern, name)]
    keep = list(dict.fromkeys(keep))  # drop duplicates, keep order
    rows = [{"step": record["step"], **{name: record["data"].get(name) for name in keep}} for record in records]
    return pd.DataFrame(rows).groupby("step").last()


read_metrics(PLAIN_VERL_METRICS, [
    r"val-core/.*mean@1", r"critic/score/mean", r"actor/pg_loss", r"actor/grad_norm", r"actor/ppo_kl", r"actor/pg_clipfrac",
    r"actor/entropy", r"rollout_corr/kl", r"response_length/mean", r"num_turns/mean",
    r"timing_s/(gen|old_log_prob|update_actor|update_weights|step)",
]).T

`trainer.rollout_data_dir` and `trainer.validation_data_dir` make verl dump every episode it generated as JSONL: the decoded prompt and response (tool calls and tool outputs included), the ground truth and the score. Here's one validation episode from before training, exactly as `ToolAgentLoop` produced it:

In [ ]:
val_before_training = [json.loads(line) for line in (WORK / "verl_val" / "0.jsonl").read_text().splitlines()]
print(f"{len(val_before_training)} validation episodes before training, mean score {np.mean([episode['score'] for episode in val_before_training]):.2f}\n")
example = val_before_training[0]
print("…" + example["input"][-200:], "\n" + "─" * 100 + "\n" + example["output"], f"\n{'─' * 100}\nground truth {example['gts']} | score {example['score']}")
for path in sorted((WORK / "verl_rollouts").glob("*.jsonl"), key=lambda path: int(path.stem)):  # one file per training step
    scores = [json.loads(line)["score"] for line in path.read_text().splitlines()]
    print(f"training step {path.stem}: {len(scores)} episodes, mean score {np.mean(scores):.2f}")

Validation accuracy goes from 0 to 1 in three steps, and almost all of it is **format**. In the validation episode above, the model calls the tool correctly, gets the right number back, and then states it in plain text instead of `\boxed{}`, so the reward is 0. Every validation episode starts that way, and the training scores (0.05 → 0.15 → 0.79) show the model learning to box its answer. The tool already does the arithmetic, so the format is all there is to learn: the lesson from §2.5, in its most extreme form.

Also notice that `actor/ppo_kl` and `actor/pg_clipfrac` are exactly 0. That's the units note above: each step is a single mini-batch of 16 prompts × 8 samples, so every update happens at ρ = 1, where the clip can't act (§1.7).

**Reading the metrics.** verl logs a few hundred metrics per step; every one in the table above is something we computed by hand earlier:

| Metric | What it tells us | Where we built it |
|---|---|---|
| `val-core/…/mean@1` | accuracy on held-out problems (greedy): the number we actually care about | `evaluate` (§1.6) |
| `critic/score/mean` | mean training reward (verl logs it under `critic/` even without a critic) | `reward` in `rl_step` (§1.6) |
| `actor/pg_loss`, `actor/grad_norm` | the update itself. The loss *value* means little; its gradient carries the signal | `ppo_update` (§2.1) |
| `actor/ppo_kl`, `actor/pg_clipfrac` | how far the policy moved within the step, and the fraction of tokens whose gradient was clipped. Both should stay small | the ratio and clip in `ppo_clip_loss` (§1.7) |
| `actor/entropy` | how spread out the policy's next-token distribution is. Collapsing entropy means the model has stopped exploring | |
| `rollout_corr/kl` | trainer-vs-vLLM log-prob mismatch. Logged when the batch carries vLLM's log-probs, as Agent Lightning's always do (§9.2) | `logp_mismatch` (§2.6) |
| `response_length/mean`, `num_turns/mean` | how the model's behaviour is changing | |
| `timing_s/*` | where the time goes: generation, old log-probs, update, weight sync | the timings in every `rl_step` |

### 9.2 verl + Agent Lightning: the real agent

Finally, the agent from §7.3 and §8 with the whole real stack: the Gateway and Controller we met in §8.3, and Agent Lightning's trainer, `AgentLightningRayPPOTrainer`, instead of our loop. That trainer is verl's `RayPPOTrainer` with one change: its `generate_sequences` step is replaced by the rollout path from §8.3 (queue rollouts, wait for the agents, build the batch with `RolloutAdapter`). This is the same command `examples/calc_x/run_local.sh` uses, with a few short-run overrides.

In [ ]:
AGL_PORT, AGL_KEY = 8283, "rl-from-scratch"
AGL_URL = f"http://127.0.0.1:{AGL_PORT}"
start_service("agl_gateway", [
    sys.executable, "-m", "agentlightning.server", "host=127.0.0.1", f"port={AGL_PORT}", f"key={AGL_KEY}",
    f"default_proxy.model_name={CALC_X_MODEL}", f"hydra.run.dir={WORK}/hydra/gateway", "hydra.output_subdir=null",
])
wait_until_healthy("agl_gateway", f"{AGL_URL}/healthz")
start_service("agl_controller", [
    sys.executable, "-m", "agentlightning.controller", "runner_type=local", f"agl_server.url={AGL_URL}", f"agl_server.key={AGL_KEY}",
    "local_runner.poll_interval=1", "local_runner.maximum_size=128", f"hydra.run.dir={WORK}/hydra/controller", "hydra.output_subdir=null",
], cwd=REPO / "examples/calc_x")
time.sleep(3)
assert SERVICES["agl_controller"].poll() is None, log_tail("agl_controller")

AGL_METRICS = WORK / "agl_metrics.jsonl"
AGL_METRICS.unlink(missing_ok=True)
start_service("agl_trainer", [
    sys.executable, "train_calc_agent.py", "--agl-base-url", AGL_URL, "--agl-key", AGL_KEY, "--model", CALC_X_MODEL, "--run-name", "from_scratch",
    "--train-file", str(CALC_X_DATA / "train.parquet"), "--val-file", str(CALC_X_DATA / "test_mini.parquet"),
    "trainer.total_training_steps=3", "data.train_batch_size=8", "actor_rollout_ref.actor.ppo_mini_batch_size=8",
    "trainer.val_before_train=True", "trainer.save_freq=-1", "trainer.logger=[console,file]",
], cwd=REPO / "examples/calc_x", env={"VERL_FILE_LOGGER_PATH": str(AGL_METRICS), "WANDB_MODE": "disabled"})
exit_code = follow_log("agl_trainer", r"Initial validation|Final validation|Training Progress|AglRolloutManager: completed=(\d+)/\1 |step:\d+ ")
print(f"\ntrainer exited with code {exit_code}")
if exit_code != 0:
    raise RuntimeError(log_tail("agl_trainer", 60))

In [ ]:
read_metrics(AGL_METRICS, [
    r"val/reward", r"training/reward", r"training/n_groups", r"training/n_zero_adv_groups",
    r"training/n_sample_collected", r"training/n_sample_dropped/same_reward", r"training/n_sample_trained",
    r"actor/pg_loss", r"actor/grad_norm", r"actor/ppo_kl", r"actor/pg_clipfrac", r"rollout_corr/kl", r"response_length/mean",
    r"timing_s/(gen|old_log_prob|update_actor|update_weights|step)",
]).T

Agent Lightning adds a few metrics of its own, and they tell the story of this run. `training/n_zero_adv_groups` counts the groups with no signal (§1.5): at every step, 5 to 7 of the 7 or 8 groups had all four attempts get the same reward, so most of each batch couldn't teach anything. `n_sample_dropped/same_reward` counts the rows dropped to fit the batch to whole mini-batches, same-reward groups first, since they cost compute and carry no gradient. That leaves exactly one mini-batch per step, so every update happens at ρ = 1 and `actor/ppo_kl` and `actor/pg_clipfrac` stay at 0, as in §9.1. `rollout_corr/kl` is about as small as our own `logp_mismatch` from §2.6.

`val/reward` went from 0.50 to 0.40, but on 20 validation questions that's well within the noise: the full Calc-X recipe runs 512 steps of 32 questions. Here the point is the plumbing.

If we compare the two runs' configs, the verl side is almost the same. What changed is *where rollouts come from*: in §9.1 verl ran a token-level agent loop inside its Ray workers; in §9.2 it registered its vLLM replicas with the Gateway, queued rollouts, and the Controller ran the unmodified AutoGen agent as ordinary processes (or Kubernetes pods) that never knew they were being trained.

In [ ]:
for name in ["agl_trainer", "agl_controller", "agl_gateway"]:
    stop_service(name)
import shutil

shutil.rmtree(SHM, ignore_errors=True)
print("remaining services:", [name for name, proc in SERVICES.items() if proc.poll() is None])

## 10. What we built, and what each library is for

| We wrote | It became | Why the library version |
|---|---|---|
| `hf_generate` | vLLM | generation speed: paged KV cache, continuous batching, CUDA graphs |
| `save_for_inference` + `reload_weights` | verl's checkpoint engine (CUDA IPC / NCCL, bucketed) | no round trip through host memory; works across GPUs and nodes |
| dict of padded tensors | `DataProto` | splits, concatenates and travels between processes |
| `grpo_advantages`, `ppo_clip_loss` | `core_algos` | the same maths, plus dual-clip, many estimators and losses, aggregation modes |
| one process | Ray actors | separate roles, resources and lifetimes |
| `MyWorkerGroup` | `RayWorkerGroup` + `@register(Dispatch...)` | one worker per GPU, N GPUs, ranks and rendezvous handled |
| `run_episodes` | `ToolAgentLoop` | multi-turn rollouts inside verl, for agents written in verl's terms |
| the 60-line gateway + `Popen` loop | Agent Lightning's Gateway + Controller | rollout state and failures, concurrency, Kubernetes, weight-sync coordination |
| `gateway_batch` | `RolloutAdapter` + `compute_rollout_level_advantage` | the same rows and advantages, plus merging prefix-continuous calls into one row |
| our training loop | `RayPPOTrainer` (and `AgentLightningRayPPOTrainer` for real agents) | all of the above, wired together and configured from one tree |

**When to use what:**

* **Single-turn**, or an agent we're happy to (re)write as a verl agent loop with verl tools → **verl alone** (§9.1).
* An **existing agent**: its own framework (AutoGen, LangChain, OpenAI Agents SDK…), its own tools and sandboxes, its own processes or containers, or code we don't control → **verl + Agent Lightning** (§9.2). The agent only changes its base URL.

The hard parts of agent RL that §7.3 and §8 made concrete (exact tokens from HTTP traffic, per-rollout bookkeeping, rollouts as processes) are exactly the parts Agent Lightning owns. Everything below that line is still verl.

---

**Next:** [notebook 4](4_rewards_and_judges.ipynb) looks at the reward itself: what an exact check really measures, and what changes when a model does the grading. [Notebook 5](5_calc_x_end_to_end.ipynb) drives one Calc-X training step through Agent Lightning by hand, then launches a real training run.